In [ ]:
# 1. サンプルデータセットをダウンロード
%cd /content
!git clone https://github.com/wal-afk/drive_sim
%cd drive_sim
!git pull
!git restore .
!git clean -fd
%cd /content/drive_sim

!pip install -U plotly==6.9

In [ ]:
import yaml

from sim.drive_simulator import CarSim, Commander
from sim.vehicle import VehicleProp
from sim.mission_base import MissionBase
from sim.goal import GoalLine, GoalCircle
from sim.drawer import SimDrawer, MissionDrawer
from sim.worlds.type_b_world import type_b_circuit
from sim.sign import Sign

with open("config/type-b.yaml", "r") as f:
    vehicle_config = yaml.safe_load(f)

prop = VehicleProp(**vehicle_config)

# プログラムの書き方講座１
## 1.プログラムは命令を上から順番に書く

例えば、命令を日本語で書いてみると・・・
```
まっすぐ2m進め
左に90度回転せよ
まっすぐ2m進め
左に90度回転せよ
まっすぐ2m進め
```
ロボットがどんな動きをするかイメージできるかな？

## 2. プログラム言語では、使える命令が決まっている

さきほどのプログラムを、プログラム言語で書くと・・・
```
run(2.0)
turn(90)
run(2.0)
turn(90)
run(2.0)
```

注：　使える命令は、使用するライブラリ等によって変わる

命令は、
```
関数名(指示値)
```
の形式で書く。上記の例の場合は、runやturnが関数名で、2.0や90が指示値。

関数にどのような指示値が使えるかは、関数によって異なる。指示値は0個や複数個の場合もあるので、よく説明を読んでからプログラムをする必要があるよ。

```
関数名()
関数名(指示値1,指示値2,指示値3)
```

**下のチュートリアルの説明をよく読んで、問題に取り組んでみよう。**

# チュートリアル1

下記の命令を組み合わせてプログラムを書き、ロボットを1m先にあるチェックポイント(goal1)を超えてから、元の位置(goal2)に戻らせよう。

## 取り組み方
1. 使える命令を理解する
2. 下のセルを実行して、ロボットの限界速度や、ロボットが存在する初期位置やチェックポイント（goal）を把握する
3. ２つ下のセル内にプログラムを書き実行して結果を見る

|使える命令|意味|指定できる値|使い方|
|--|--|--|--|
|move|一定速度で前に進む|v=速度[m/秒]|move(v=0.2)|
|move|一定時間だけ一定速度で前に進む|v=速度[m/秒], t=時間[秒]|move(v=0.2, t=1.0)|
|rotate|一定速度で回転する|w=回転速度[度/秒]|rotate(w=90)|
|rotate|一定時間だけ一定速度で回転する|w=回転速度[度/秒], t=時間[秒]|rotate(w=90, t=1.0)|
|wait|直前の命令が終わるまで待つ||wait()|

## 注意点
- 命令は直後にwaitを置かない場合、どんどん下に実行されていってしまう。
    - 「一定時間だけ〇〇する」を最後までに実行するには、wait命令を直後に置く必要がある
- スタート時の位置はランダムに最大10cmほどずれる
- スタート時の向きはランダムに最大10度ほどずれる

In [ ]:
class Tutorial1Base(MissionBase):
    def __init__(self):
        super().__init__(type_b_circuit, t_max=20)
        self.goals = [
            GoalLine((1.0, 0.0), should_stop=False),
            GoalCircle((0.0, 2.0), 0.2),
        ]
        self.initial_xy = (0.0, 2.0)
        self.random_d_xy = (0.1, 0.1)
        self.random_d_yaw_deg = 10

print("最大速度", prop.max_velocity, "m/秒")
print("最大回転速度", prop.max_rotate_deg, "度/秒")
MissionDrawer(Tutorial1Base()).show()

In [ ]:
class Tutorial1(Tutorial1Base):
    @staticmethod
    def command_func(*, move, rotate, wait, **kwargs):
        ######## ここから下に正しいプログラムを書こう
        move(v=0.2, t=3)  # 書き方の例
        wait()
        ######## ここより上に正しいプログラムを書こう
        ######## プログラムを書いた後にセルを実行し結果を確認しよう


sim = CarSim(prop, Tutorial1())
success = sim.run()
if success:
    SimDrawer(sim).show()

# チュートリアル1のヒント

1. 進む距離[m]は、速度[m/秒] × 時間[秒] になる。回転する角度[度]は、回転速度[度/秒] × 時間[秒] になる。進みたい距離から、どれぐらいの速度と時間で進むと良いかを考えよう
2. スタートの位置や角度はランダムに少しずれるので、そのことも考慮して進みたい距離を考えよう。チェックポイントにまっすぐ進めない(斜めに進んでしまう)場合、進む距離が1mだと届かないかも？

# プログラムの書き方講座2

## 1.変数って何？

- 「〇〇を見つけろ」のような命令は、見つけた結果（値）を返してくれる
  - 値を記録しておいて、後で使えるようにする為の箱のようなものを「変数」と呼ぶ
  - 箱には好きな名前をつけられる（例えばposのように）

- 下記であれば、虫を見つけた結果をposという名前で記録しておき後で利用できるようにする、という命令になる
```
pos = search(name="bug")
```

記録した値に何が入っているか？どう使うか？は関数によって異なるよ。
例えばsearchの結果の場合、pos.xと書くと「見つけた虫の前方距離」を表すといったように決まっている。

## 四則演算（足し算・引き算・掛け算・割り算）

- 数値が入っている変数や数字の間では足し算・引き算・掛け算・割り算などの計算ができる
- 足し算は`+`、引き算は`-`、掛け算は`*`、割り算は`/`で書くよ
- 例えば下記であれば、見つけた虫までの前方距離の長さだけ速度0.2[m/秒]で進むという意味になる
  - 注：速度が決まっている時、何秒進めばよいかは「進みたい距離 ÷ 速度」で求まる
```
pos = search(name="bug")
move(v=0.2, t=pos.x/0.2)
```


**下のチュートリアルの説明をよく読んで、問題に取り組んでみよう。**

# チュートリアル2

下記の命令を組み合わせてプログラムを書き、ロボットを回転させて標識(標識名はsign1)が正面にくる位置で止まってから、直進して標識までのちょうど半分の距離まで進んで止まろう。
標識は開始時点でカメラの視界内に必ずあるよ。

## 取り組み方
1. 使える命令を理解する
    - move,rotate,waitはチュートリアル1と同じ。searchが追加。
2. 下のセルを実行して、ロボットの限界速度や、ロボットが存在する初期位置やチェックポイント（goal）を把握する
3. ２つ下のセル内にプログラムを書き実行して結果を見る

|使える命令|意味|指定できる値|使い方|
|--|--|--|--|
|move|一定速度で前に進む|v=速度[m/秒]|move(v=0.2)|
|move|一定時間だけ一定速度で前に進む|v=速度[m/秒], t=時間[秒]|move(v=0.2, t=1.0)|
|rotate|一定速度で回転する|w=回転速度[度/秒]|rotate(w=90)|
|rotate|一定時間だけ一定速度で回転する|w=回転速度[度/秒], t=時間[秒]|rotate(w=90, t=1.0)|
|wait|直前の命令が終わるまで待つ|-|wait()|
|search|標識を見つける（複数見つかった場合は、最も近いもの）|-|pos = Search()|
|search|特定の標識を見つける（複数見つかった場合は、最も近いもの）|name = "標識名"|pos = Search(name="sign1")|

- pos = search() が返す値には下記が含まれる
  - pos.x: 見つけた標識の前方位置[m]　※前方が正
  - pox.y: 見つけた標識の左右位置[m]　※左側が正、右側は負
  - pos.r: 見つけた標識への距離[m]
  - pos.theta: 見つけた標識の角度[度]　※左側が正、右側は負
  - pos.name: 見つけた標識の標識名

## 注意点
- スタート時の向きはランダムに最大5度ほどずれる

In [ ]:
class Tutorial2Base(MissionBase):
    def __init__(self):
        super().__init__(type_b_circuit, t_max=20)
        self.goals = [
            GoalCircle((0.6, 1.75), 0.1, should_stop=True),
        ]
        self.initial_xy = (0.0, 2.0)
        self.random_d_yaw_deg = 5
        self.set_signs(
            [
                Sign(x=1.2, y=1.5, name="sign1"),
            ]
        )


print("最大速度", prop.max_velocity, "m/秒")
print("最大回転速度", prop.max_rotate_deg, "度/秒")
MissionDrawer(Tutorial2Base()).show()

In [ ]:
class Tutorial2(Tutorial2Base):
    @staticmethod
    def command_func(*, move, rotate, search, wait, **kwargs):
        ######## ここから下に「標識の方を向く」プログラムを書こう
        pos = search()
        rotate(w=90.0, t=1.0)  # 書き方の例
        wait()
        ######## ここより上に「標識の方を向く」プログラムを書こう

        ######## ここから下に「標識までの半分の距離だけ前進する」プログラムを書こう
        pos = search()
        move(v=0.2, t=3.0)  # 書き方の例
        wait()
        ######## ここより上に「標識までの半分の距離だけ前進する」プログラムを書こう
        ######## プログラムを書いた後にセルを実行し結果を確認しよう


sim = CarSim(prop, Tutorial2())
success = sim.run()
if success:
    SimDrawer(sim).show()

# プログラムの書き方講座3

## 1. 条件に応じて動作を変えよう

- プログラムは、特定の条件が成立するかしないかによって、実行する命令を変えることができる
  - これを「条件分岐」と呼ぶ 
- 例えば「虫を見つけたら回転せよ」というプログラムを日本語で書くと
  ```
  虫を見つけよ
  ・もしみつけてないなら
    ・停止せよ
  ・もし見つけたなら
  　・回転せよ
  ```
- 上記をプログラミング言語(python)では下記のように書ける
  ```
  pos = search()
  if pos is None:
    rotate(w=0)
  else:
    rotate(w=90)
  ```

- 解説
  - `if 条件文:` で、「もし、条件文が成り立つなら」という意味になる
    - 最後の:も忘れないように
  - `else:`で「条件文が成り立たないなら」という意味になる
    - `else:`の中に書きたい命令がない場合、`else:`は書かなくてよい
  - 命令は、`if 条件文:`や`else:`の下に段落をつくって（左側にスペースを入れて）かく
  - 条件文は、「値　比較記号　値」の形で書く
    - 比較記号には >, <, >=, <=, ==, !=, is, is not　等が使える
    - 値には「変数」「数値」「文字列」「None」等が使える
    - 条件の例１：　pos is None 
        - 「posがない場合」という条件になる
    - 条件の例２：　pos is not None 
        - 「posがある場合」という条件になる
    - 条件の例３：　pos.theta > 10
        - 「pos.thetaが10より大きい場合」という条件になる
    - 条件の例４：　10 < pos.theta
        - 例３と同じ意味
    - 条件の例５：　pos.theta == 0 
        - 「pos.thetaが0の場合」という条件になる
    - 条件の例６：　pos.name == "bug" 
        - 「pos.nameがbugである場合」という条件になる

## 2. より複雑な条件

- 分岐は多段階にも書ける
  - 例えば「条件文１が成り立たない時に、さらに条件文２が成り立つかどうかで命令を分岐したい」場合は下記のように書ける
    ```
    if 条件文１:
      命令()
      命令()
      ・・・略
    else:
      if 条件文２:
        命令()
        命令()
        ・・・略
      else:
        命令()
        命令()
        ・・・略
    ```
- 組み合わせの条件文「〇〇かつ△△」や「〇〇もしくは△△」等を書きたい場合
  - `条件文1 and 条件文2`と書くと、「条件文1と条件文2の両方が正しい場合に成り立つ」という条件文になる
  - `条件文1 or 条件文2`と書くと、「条件文1もしくは条件文2のうち少なくとも１つは正しい場合に成り立つ」という条件文になる
  - `not 条件文1`と書くと、「条件文1が正しくない場合に成り立つ」という条件文になる
- 例えば、「ほぼ正面に虫を見つけたら、虫の位置まで前進する」というプログラムを日本語で書くと
  ```
  虫を見つけよ
  ・もし見つけたなら
  　・もし虫がほぼ正面（±5度以内）なら
  　　・虫の位置まで前進せよ
  ```
- 上記をプログラミング言語に直すと下記になる
  ```
  pos = search()
  if pos is None:
    move(v=0)
  else:
    if pos.theta <= 5 and pos.theta >= -5:
      move(v=0.2, t=pos.x/0.2)
      wait()
    else:
      move(v=0)
  ```


## 3.プログラムを繰り返そう

- 上記の「ほぼ正面に虫を見つけたら、虫の位置まで前進する」プログラムだと虫を探す処理を１度だけして、その結果に応じて一度だけ処理を行うとプログラムが終了してしまう。「ほぼ正面に虫を見つけたら、虫の位置まで前進する」という処理をずっとし続けたい（いつ虫が来てもいいように）という場合は、下記のように書くことで繰り返し同じプログラムを実行できる。
  - 繰り返したい処理は、段落をつくって（左側にスペースを入れて）かく
  ```
  while True:
    繰り返したい処理
  ```
- 「ほぼ正面に虫を見つけたら、虫の位置まで前進する」をずっとしたい場合は以下のように書ける
  ```
  while True:
    pos = search()
    if pos is None:
      move(v=0)
    else:
      if pos.theta <= 5 and pos.theta >= -5:
        move(v=0.2, t=pos.x/0.2)
        wait()
      else:
        move(v=0)
  ```

# チュートリアル3

下記の命令を組み合わせてプログラムを書き、ロボットを直進させながら標識(標識名はsign1)が車にぶつかりそう（左右0.2[m]以内）に見えた瞬間に停止しよう。

## 取り組み方
1. 使える命令を理解する
    - チュートリアル2と同じだがrotateは使用不可
2. 下のセルを実行して、ロボットの限界速度や、ロボットが存在する初期位置やチェックポイント（goal）を把握する
3. ２つ下のセル内にプログラムを書き実行して結果を見る

|使える命令|意味|指定できる値|使い方|
|--|--|--|--|
|move|一定速度で前に進む|v=速度[m/秒]|move(v=0.2)|
|move|一定時間だけ一定速度で前に進む|v=速度[m/秒], t=時間[秒]|move(v=0.2, t=1.0)|
|wait|直前の命令が終わるまで待つ|-|wait()|
|search|標識を見つける（複数見つかった場合は、最も近いもの）|-|pos = Search()|
|search|特定の標識を見つける（複数見つかった場合は、最も近いもの）|name = "標識名"|pos = Search(name="sign1")|

- pos = search() が返す値には下記が含まれる
  - pos.x: 見つけた標識の前方位置[m]　※前方が正
  - pox.y: 見つけた標識の左右位置[m]　※左側が正、右側は負
  - pos.r: 見つけた標識への距離[m]
  - pos.theta: 見つけた標識の角度[度]　※左側が正、右側は負
  - pos.name: 見つけた標識の標識名

## 注意点
- スタート時の位置はランダムに前後最大30cmほどずれる（左右ずれはない）

In [ ]:
class Tutorial3Base(MissionBase):
    def __init__(self):
        super().__init__(type_b_circuit, t_max=20)
        self.goals = [
            GoalCircle((2.2, 0.0), 0.2, should_stop=True),
        ]
        self.initial_xy = (0.0, 0.0)
        self.random_d_xy = (0.3, 0.0)
        self.set_signs(
            [
                Sign(x=1.9, y=-0.4, name="sign1"),
                Sign(x=2.7, y=0.4, name="sign1"),
                Sign(x=3.5, y=-0.1, name="sign1"),
            ]
        )


print("最大速度", prop.max_velocity, "m/秒")
print("最大回転速度", prop.max_rotate_deg, "度/秒")
MissionDrawer(Tutorial3Base()).show()

In [ ]:
class Tutorial3(Tutorial3Base):
    @staticmethod
    def command_func(*, move, search, wait, **kwargs):
        # ヒントとして、「searchして標識が見つからなかったら0.1[m]だけ前進する」を繰り返すという処理を記載済み
        # 「searchして標識が見つかった場合」のプログラムだけを書けばOK

        while True:
            pos = search(name="sign1")
            if pos is None:
                move(v=0.2, t=0.5)
                wait()
            else:
                ######## ここから下に「標識が左右0.2[m]以内でなければ0.1[m]だけ前進する」プログラムを書こう
                move(v=0.2, t=0.5)
                wait()
                ######## ここより上にプログラムを書こう
        ######## プログラムを書いた後にセルを実行し結果を確認しよう

sim = CarSim(prop,Tutorial3())
success = sim.run()
if success:
    SimDrawer(sim).show()

# プログラムの書き方講座4

## スムーズな動作にしよう

- チュートリアル3では、標識をチェックしながら車を前進させるために、下記のように繰り返していた
  - 標識をチェックする
  - 標識がぶつかりそうにないなら10cm進む
  - 標識をチェックする
  - 標識がぶつかりそうにないなら10cm進む
  - ・・・(繰り返し)
- この方法には問題がある。どんな問題が思いつく？

- スムーズに動かすには下記の方が望ましい
    ```
    標識を見つけよ
    - 標識にぶつかりそうなら止まれ
    - 標識にぶつかりそうにないなら決まった前進速度を維持
    最初に戻って繰り返せ
    ```
- こうしておくと、コンピューターの能力を最大限活用した速さで標識に気付いて止まることができる

## プログラムを改善すると・・・

- 下記の「ほぼ正面に虫を見つけたら、虫の方向に前進する」プログラムをスムーズに動作するように書くと下記になる。何が違うか比べてみよう。
- 改善前　：　ほぼ正面に虫がいる限り0.2[m]づつ前進し続ける
  ```
  while True:
    pos = search()
    if pos is None:
      move(v=0)
    else:
      if pos.theta <= 5 and pos.theta >= -5:
        move(v=0.2, t=1.0)
        wait()
      else:
        move(v=0)
  ```
- 改善後　：　ほぼ正面に虫がいる限り、0.2[m/秒]で前進し続ける
  ```
  while True:
    pos = search()
    if pos is None:
      move(v=0)
    else:
      if pos.theta <= 5 and pos.theta >= -5:
        move(v=0.2)
      else:
        move(v=0)
  ```

- ポイント
    - 0.2[m]進むには時間がかかる（1秒程度）のに対して、速度を0.2[m/秒]に設定するのは一瞬で終わる
    - wait()を使わないことで、すぐに次の繰り返しが始まる
        - コンピュータの処理能力によるが、例えば10分の1秒～100分の1秒の速さで認識に応じた車のコントールもできる


# チュートリアル4

下記の命令を組み合わせてプログラムを書き、ロボットを直進させながら標識(標識名はsign1)が車にぶつかりそう（左右0.2[m]以内）に見えた瞬間に停止しよう。※問題はチュートリアル3と同じ

## 取り組み方
1. 使える命令を理解する
    - チュートリアル3と同じだがwaitは使用不可
2. 下のセルを実行して、ロボットの限界速度や、ロボットが存在する初期位置やチェックポイント（goal）を把握する
3. ２つ下のセル内にプログラムを書き実行して結果を見る

|使える命令|意味|指定できる値|使い方|
|--|--|--|--|
|move|一定速度で前に進む|v=速度[m/秒]|move(v=0.2)|
|search|標識を見つける（複数見つかった場合は、最も近いもの）|-|pos = Search()|

- pos = search() が返す値には下記が含まれる
  - pos.x: 見つけた標識の前方位置[m]　※前方が正
  - pox.y: 見つけた標識の左右位置[m]　※左側が正、右側は負
  - pos.r: 見つけた標識への距離[m]
  - pos.theta: 見つけた標識の角度[度]　※左側が正、右側は負
  - pos.name: 見つけた標識の標識名

## 注意点
- スタート時の位置はランダムに前後最大30cmほどずれる（左右ずれはない）

In [ ]:
class Tutorial4Base(MissionBase):
    def __init__(self):
        super().__init__(type_b_circuit, t_max=20)
        self.goals = [
            GoalCircle((2.2, 0.0), 0.2, should_stop=True),
        ]
        self.initial_xy = (0.0, 0.0)
        self.random_d_xy = (0.3, 0.0)
        self.set_signs(
            [
                Sign(x=1.9, y=-0.4, name="sign1"),
                Sign(x=2.7, y=0.4, name="sign1"),
                Sign(x=3.5, y=-0.1, name="sign1"),
            ]
        )


print("最大速度", prop.max_velocity, "m/秒")
print("最大回転速度", prop.max_rotate_deg, "度/秒")
MissionDrawer(Tutorial4Base()).show()

In [ ]:
class Tutorial4(Tutorial4Base):
    @staticmethod
    def command_func(*, move, search, **kwargs):
        # ヒントとして、「searchして標識が見つからなかったら前進する」を繰り返すという処理を記載済み
        # 「searchして標識が見つかった場合」のプログラムだけを書けばOK

        while True:
            pos = search(name="sign1")
            if pos is None:
                move(v=0.2)
            else:
                # ####### ここから下に「標識が左右0.2[m]以内なら停止し、そうでないなら一定速度で前進する」プログラムを書こう
                move(v=0.0)
                # ####### ここより上にプログラムを書こう
        # ####### プログラムを書いた後にセルを実行し結果を確認しよう


sim = CarSim(prop, Tutorial4())
success = sim.run()
if success:
    SimDrawer(sim).show()